In [ ]:
# Escanea el mercado y trae los tickers cuyo precio este N% de su média móvil ponderada de 150 y 200 en SEMANAL.
# Correr sábado o domingo: todas las barras son semanas cerradas

In [ ]:
#Importa las librerias y módulos necesarios.
%load_ext autoreload
%autoreload 2
import pandas as pd
from datos import obtener_tickers

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
# Descarga de datos semanales.

data_7d = obtener_tickers("1wk", "max")

Se han cargado 530 tickers...
Tiempo de descarga en 6886.29 segundos.
Se descargó datos en temporalidad1wk con un periodo de max


In [19]:
resultados = []

for ticker in data_7d.columns.get_level_values(0).unique():
    df_ticker = data_7d[ticker].dropna(subset=['Close', 'High', 'Low'])
    # Descartar la vela semanal en curso
    df_ticker = df_ticker.iloc[:-1]
    
    if len(df_ticker) >= 600:
        cierre = df_ticker['Close']
        high = df_ticker['High']
        
        s_ema_150 = cierre.ewm(span=150, adjust=False).mean()
        s_ema_200 = cierre.ewm(span=200, adjust=False).mean()
        
        ema_150 = s_ema_150.iloc[-1].item()
        ema_200 = s_ema_200.iloc[-1].item()
        precio = cierre.iloc[-1].item()

        # Condición 1: Estructura alcista y pendiente positiva en ambas medias
        pendiente_150 = ema_150 > s_ema_150.iloc[-14].item()
        pendiente_200 = ema_200 > s_ema_200.iloc[-14].item()
        estructura_valida = (ema_150 > ema_200) and pendiente_150 and pendiente_200

        if estructura_valida:
            dist_150 = ((precio - ema_150) / ema_150) * 100
            dist_200 = ((precio - ema_200) / ema_200) * 100

            # Condición 2: El precio está tocando la zona de la EMA (a 1% o menos)
            toque_150 = abs(dist_150) <= 1.0
            toque_200 = abs(dist_200) <= 1.0

            # Condición 3 (Antirrebote de techo): Validar que el toque sea DESDE ARRIBA.
            # Verificamos que en las últimas 8 semanas el precio haya estado operando
            # claramente por encima de la zona, garantizando que viene corrigiendo.
            max_previo_150 = high.iloc[-8:-1].max() > (ema_150 * 1.03)
            max_previo_200 = high.iloc[-8:-1].max() > (ema_200 * 1.03)

            soportes_validos = (toque_150 and max_previo_150) or (toque_200 and max_previo_200)

            if soportes_validos:
                resultados.append({
                    'ticker': ticker,
                    'precio': precio,
                    'dist_150': dist_150,
                    'dist_200': dist_200,
                })

for r in resultados:
    print(
        f"SOPORTE | Ticker: {r['ticker']} | Precio: {r['precio']:.2f} | "
        f"Dist. EMA 150: {r['dist_150']:.2f}% | Dist. EMA 200: {r['dist_200']:.2f}%"
    )

SOPORTE | Ticker: URA | Precio: 34.11 | Dist. EMA 150: -7.58% | Dist. EMA 200: 0.10%
SOPORTE | Ticker: VMC | Precio: 261.69 | Dist. EMA 150: 0.88% | Dist. EMA 200: 5.49%
SOPORTE | Ticker: CBRE | Precio: 123.30 | Dist. EMA 150: -4.66% | Dist. EMA 200: 0.99%
SOPORTE | Ticker: ICLN | Precio: 16.02 | Dist. EMA 150: -1.69% | Dist. EMA 200: -0.20%
SOPORTE | Ticker: HII | Precio: 274.08 | Dist. EMA 150: -0.44% | Dist. EMA 200: 4.34%
SOPORTE | Ticker: F | Precio: 11.60 | Dist. EMA 150: -0.33% | Dist. EMA 200: 3.19%
SOPORTE | Ticker: GPC | Precio: 123.02 | Dist. EMA 150: 0.56% | Dist. EMA 200: 1.32%
SOPORTE | Ticker: TTWO | Precio: 203.69 | Dist. EMA 150: 0.14% | Dist. EMA 200: 4.99%
